In [ ]:
# --- Visualization Notebook: Fused Record Inspector ---

from pathlib import Path

import ipywidgets as widgets
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from PIL import Image as PILImage

In [ ]:
# Path to fused JSONL (adjust as needed)
DATA_PATH = Path("../data/fused/bagel_001.jsonl")

# Load fused data
df = pd.read_json(DATA_PATH, lines=True)

records_flat = []
for record in df.to_dict(orient="records"):
    for reading in record["sensor_window"]:
        flat = {**record, **reading}  # merge window metadata with reading
        records_flat.append(flat)

df_flat = pd.DataFrame(records_flat)


print(f"Loaded {len(df_flat)} fused records.")
display(df_flat.head())

In [ ]:
df_flat.info()
df_flat.describe(include="all")

In [ ]:
fig = go.Figure()
for axis in ["x", "y", "z"]:
    fig.add_trace(
        go.Scatter(
            x=df_flat["timestamp_ns"], y=df_flat[f"force_{axis}"], name=f"Force {axis.upper()}"
        )
    )
for axis in ["x", "y", "z"]:
    fig.add_trace(
        go.Scatter(
            x=df_flat["timestamp_ns"], y=df_flat[f"torque_{axis}"], name=f"Torque {axis.upper()}"
        )
    )
fig.update_layout(title="Force & Torque over Time", xaxis_title="Timestamp", yaxis_title="Value")
fig.show()

In [ ]:
pose_cols = [c for c in df_flat.columns if "joint_" in c or "pos_" in c]
if pose_cols:
    fig = px.line(df_flat, x="timestamp_ns", y=pose_cols, title="Pose / Joint Angles")
    fig.show()
else:
    print("No pose or joint angle columns found.")

In [ ]:
def show_frame(index):
    row = df_flat.iloc[index]
    path = "../" + row["rgb_path"]
    print(path)
    try:
        img = PILImage.open(path)
        display(img)
    except Exception as e:
        print(f"Failed to render frame {index}: {e}")


frame_slider = widgets.IntSlider(min=0, max=len(df_flat) - 1, step=1, description="Frame")
widgets.interact(show_frame, index=frame_slider)

In [ ]:
# Convert to seconds relative to first timestamp
df_flat["timestamp_s"] = (df_flat["timestamp_ns"] - df_flat["timestamp_ns"].iloc[0]) * 1e-9

fig = px.scatter(df_flat, x="timestamp_s", y=df_flat.index, title="Frame Timestamp Sequence (s)")
fig.update_xaxes(title="Time (s)")
fig.update_yaxes(title="Frame Index")
fig.show()

In [ ]:
print("✅ Visualization complete — dataset ready for inspection.")